# LDSC correlation & Heatmap Visualizer

## Purpose
Parses LDSC (Linkage Disequilibrium Score Regression) log files to extract genetic correlation (rg) results, mirrors the pairwise data to guarantee symmetry, calculates p-value significance markers, and visualizes the upper triangle as a publication-ready heatmap.

## Input
* LDSC `.log` files located in the target directory containing a standard "Summary of Genetic Correlation Results" block.

## Process
1. **Parse** `.log` files to extract metadata and specific metric columns.
2. **Convert** metrics to numeric types and clean phenotype string identifiers.
3. **Programmatically duplicate** rows with reversed (p2, p1) pairs to ensure complete symmetric data matrices.
4. **Pivot and reindex** datasets according to a predefined phenotype order.
5. **Construct** significance annotation string grids based on p-value buckets.
6. **Mask** the lower triangle and generate a custom-mapped correlation heatmap.

## Output
* Jupyter notebook inline displays of dataframes and annotation matrix.
* A vector-based, text-editable PDF plot: `heatmap_editable.pdf`.

In [ ]:
import os
import glob
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

In [ ]:
# ------------------------------------------------------------------------------
# 1. SETUP & CONFIGURATION
# ------------------------------------------------------------------------------

# Define your log file directory
log_dir = "/home/.../new_prs_combinations"
log_files = glob.glob(os.path.join(log_dir, "*.log"))

# Define expected columns
columns = [
    "p1", "p2", "rg", "se", "z", "p",
    "h2_obs", "h2_obs_se", "h2_int", "h2_int_se",
    "gcov_int", "gcov_int_se"
]

# Order of phenotypes for the final visualization matrix
order = ['AD', 'PD_PROXY', 'DLB', 'FTD', 'ALS', 'NDEG', 'MS', 'MSA_R', 'MDD', 'BP', 'SCZ']

# Configure matplotlib to embed editable TrueType fonts for Illustrator
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

In [ ]:
# ------------------------------------------------------------------------------
# 2. LOG FILE PARSING
# ------------------------------------------------------------------------------

results = []

for file in log_files:
    with open(file, "r") as f:
        lines = f.readlines()
        
    for i, line in enumerate(lines):
        if line.strip().startswith("Summary of Genetic Correlation Results"):
            if i + 2 < len(lines):  # Ensure the summary line exists
                summary_line = lines[i + 2].strip()
                values = summary_line.split()
                if len(values) == len(columns):
                    result = dict(zip(columns, values))
                    result["filename"] = os.path.basename(file)
                    results.append(result)
            break  # Stop processing file after first summary block match
            
# Create DataFrame
df = pd.DataFrame(results)

In [ ]:
# ------------------------------------------------------------------------------
# 3. DATA CLEANING & REGEX PROCESSING
# ------------------------------------------------------------------------------

# Convert numeric columns
numeric_cols = [col for col in columns if col not in ["p1", "p2"]]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# Normalize phenotype string identifiers
df["p1"] = df["p1"].str.replace(".sumstats.gz", "", regex=False)
df["p2"] = df["p2"].str.replace(".sumstats.gz", "", regex=False)

df["p1"] = df["p1"].str.replace("../munged_sumstats/", "", regex=False)
df["p2"] = df["p2"].str.replace("../munged_sumstats/", "", regex=False)
df["p1"] = df["p1"].str.replace("munged_sumstats/", "", regex=False)
df["p2"] = df["p2"].str.replace("munged_sumstats/", "", regex=False)

df['p1'] = df['p1'].str.upper()
df['p2'] = df['p2'].str.upper()

In [ ]:
# ------------------------------------------------------------------------------
# 4. GENERATING SYMMETRIC PAIRS
# ------------------------------------------------------------------------------

# Create a set of (p1, p2) and (p2, p1) tuples from the original df
existing_pairs = set(zip(df['p1'], df['p2']))
reversed_pairs = set((b, a) for a, b in existing_pairs)

# Identify missing reversed pairs
missing_reversed = reversed_pairs - existing_pairs

# Create new rows for the missing reversed pairs
new_rows = df[df.apply(lambda row: (row['p2'], row['p1']) not in existing_pairs, axis=1)].copy()
new_rows[['p1', 'p2']] = new_rows[['p2', 'p1']]

# Append new rows to original DataFrame
df_full = pd.concat([df, new_rows], ignore_index=True)

# Show result snippet
display(df_full[['p1', 'p2', 'se', 'rg', 'p']].head(20)) 

In [ ]:
# ------------------------------------------------------------------------------
# 5. PIVOT TABLES & REINDEXING
# ------------------------------------------------------------------------------

# Create pivot tables for matrices
rg_data = df_full[['p1', 'p2', 'rg']].pivot(index="p2", columns="p1", values="rg")
p_data  = df_full[['p1', 'p2', 'p']].pivot(index="p2", columns="p1", values="p")

# Reindex layout to match target phenotype order
rg_data = rg_data.reindex(index=order, columns=order)
p_data  = p_data.reindex(index=order, columns=order)

In [ ]:
# ------------------------------------------------------------------------------
# 6. SIGNIFCANCE ANNOTATION MATRIX GENERATION
# ------------------------------------------------------------------------------

annot = rg_data.copy().astype(str)

for row in rg_data.index:
    for col in rg_data.columns:
        p_val = p_data.loc[row, col]

        if pd.isna(p_val):
            annot.loc[row, col] = ""
        elif p_val < 0.0005:
            annot.loc[row, col] = "***"
        elif p_val < 0.005:
            annot.loc[row, col] = "**"
        elif p_val < 0.05:
            annot.loc[row, col] = "*"
        else:
            annot.loc[row, col] = ""

display(annot.head(10))  

In [ ]:
# ------------------------------------------------------------------------------
# 7. HEATMAP PLOTTING & EXPORT
# ------------------------------------------------------------------------------

# Generate mask for the lower triangle
mask = np.tril(np.ones_like(rg_data, dtype=bool))

# Set up color mapping boundaries
vmin = rg_data.min().min()   
vmax = rg_data.max().max()   

# Custom Colormap definition (kept for logical script retention)
grey = "#C97A9B"
white = "#FFFFFF"
blue = "#2A418F"
cmap_div = LinearSegmentedColormap.from_list("grey_white_blue", [grey, white, blue])

# Plotting Configuration
plt.figure(figsize=(10, 8))

sns.heatmap(
    rg_data,
    annot=annot,
    fmt="",
    cmap=sns.color_palette("Spectral_r", as_cmap=True),
    center=0,
    vmin=vmin,
    vmax=vmax,
    square=True,
    mask=mask,
    linewidths=0.5,
    cbar_kws={"label": "Genetic Correlation (rg)"}
)

plt.title("Genetic Correlation Heatmap (Upper Triangle)")
plt.xlabel("p1")
plt.ylabel("p2")
plt.tight_layout()

# Save and render
output_file = "heatmap_editable.pdf" # replace the desired path name
plt.savefig(output_file, format="pdf")
plt.show()